# UNet + Transformer baseline — inference & submission

Runs the trained baseline on the test volumes and writes `submission.csv`.

It runs with **no internet**: the model weights, repo source, and dependency wheels all come from the artifacts dataset.

> **Setup before running:**
> 1. Enable the **GPU** accelerator; leave **Internet OFF**.
> 2. **Add Input** → the competition data *and* your `cellmot-baseline-artifacts` dataset (from the train notebook's output).
> 3. Set `ARTIFACTS` below to match the dataset's mount path.

## 1. Configuration

In [ ]:
import glob
import os

COMP_SLUG = "biohub-cell-tracking-during-development"

# The competition mount point also varies; find the directory that actually
# contains a test/ subfolder of zarr datasets.
_comp_candidates = [
    p for p in glob.glob(f"/kaggle/input/*{COMP_SLUG}*", recursive=True)
    if os.path.isdir(os.path.join(p, "test"))
]
if not _comp_candidates:
    raise FileNotFoundError(
        f"Competition input not mounted. /kaggle/input contains: {os.listdir('/kaggle/input')}"
    )
COMP_DIR = sorted(_comp_candidates, key=len)[0]
TEST_DIR = f"{COMP_DIR}/test"
print("Competition data:", COMP_DIR)

# The artifacts dataset can mount at varying nesting depths, so resolve it
# dynamically by locating the bundled wheels directory.
_candidates = [
    p[: -len("/wheels")]
    for p in glob.glob("/kaggle/input/**/wheels", recursive=True)
    if os.path.isdir(p)
]
if not _candidates:
    raise FileNotFoundError(
        f"Artifacts dataset not mounted. /kaggle/input contains: {os.listdir('/kaggle/input')}"
    )
ARTIFACTS = sorted(_candidates, key=len)[0]
print("Artifacts:", ARTIFACTS)

REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"

# --- Test-time options (tune these) ---------------------------------------
# Model checkpoint (relative to the repo, or an absolute path to your own).
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"

# Detection peak threshold. GT is sparse so the detector is poorly calibrated;
# 0.5 is too low, ~0.99 scored best in a sweep.
DET_THRESHOLD = 0.99

UNET_BATCH_SIZE = 4          # frame-pairs per UNet forward; lower if you OOM
SLICE = ""                   # e.g. ":5" to predict only the first 5 videos; "" = all

# Linking. ILP = global, flow-consistent (cleaner tracks; ~0.73->0.79).
# Set False for the faster greedy linker.
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0           # weight on edge probability
ILP_APPEARANCE_WEIGHT = 0.1      # cost of a track appearing
ILP_DISAPPEARANCE_WEIGHT = 0.1   # cost of a track disappearing
ILP_DIVISION_WEIGHT = 1.0        # cost of a division; lower (~0.2) allows more splits
# --------------------------------------------------------------------------


In [ ]:
import torch

# Fail fast (seconds, before any real work) if Kaggle assigns a GPU whose
# compute capability this PyTorch build cannot run. The baseline was
# validated on a T4 (sm_75); a P100 (sm_60) crashes deep inside inference.
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    arch = torch.cuda.get_arch_list()
    print(f"GPU: {torch.cuda.get_device_name()} | capability {cap} | torch arches {arch}")
    assert cap >= (7, 0), (
        f"GPU compute capability {cap} is below the minimum supported by this "
        f"PyTorch build (sm_70). Aborting before wasting compute."
    )
else:
    print("WARNING: no GPU available - inference will be extremely slow on CPU.")


## 2. Offline install & setup

Install the dependencies from the bundled wheels (no internet), then copy the repo source and weights into a writable location and put the package on the import path.

In [ ]:
import sys
import shutil
import subprocess

subprocess.run(
    ["pip", "install", "--no-index", "--find-links", f"{ARTIFACTS}/wheels",
     "tracksdata", "zarr>=3.0.10", "pyscipopt"],
    check=True,
)

shutil.copytree(f"{ARTIFACTS}/repo", REPO_DIR, dirs_exist_ok=True)
shutil.copytree(f"{ARTIFACTS}/weights", f"{REPO_DIR}/weights", dirs_exist_ok=True)
sys.path.insert(0, f"{REPO_DIR}/src")

print("Weights:", os.listdir(f"{REPO_DIR}/weights/{METHOD}/split_0"))

## 3. Inference on all test videos

Build a one-fold splits file listing every test video, then run prediction. `PYTHONPATH=src` makes the `tracking_cellmot` package importable without an internet install. Each video's tracks are exported as a `.geff` graph.

In [ ]:
import json

test_stems = sorted(f[:-5] for f in os.listdir(TEST_DIR) if f.endswith(".zarr"))
print(f"{len(test_stems)} test videos")

with open(f"{REPO_DIR}/kaggle_test_splits.json", "w") as f:
    json.dump([{"split": 0, "train": [], "test": test_stems}], f)

## 3. Inference — candidate graphs for a grid of detection thresholds


In [ ]:
import importlib.util
from pathlib import Path

import torch

# Import the prediction module directly so we can run inference once and
# reuse the candidate graphs across many ILP settings (CPU-only).
spec = importlib.util.spec_from_file_location(
    "put", f"{REPO_DIR}/scripts/predict_unet_transformer.py"
)
put = importlib.util.module_from_spec(spec)
spec.loader.exec_module(put)

from tracking_cellmot.io import save_graph

device = torch.device("cuda")
model, window_size, downsample = put.load_model(Path(REPO_DIR) / WEIGHTS, device)

DET_GRID = [0.95, 0.99]

candidate_geffs = {}  # det_thr -> {name: geff_path}
for det_thr in DET_GRID:
    cfg = put.PredictConfig(det_threshold=det_thr, use_ilp=True)
    out_dir = Path(REPO_DIR) / "candidates" / f"det_{det_thr:.3f}"
    out_dir.mkdir(parents=True, exist_ok=True)
    candidate_geffs[det_thr] = {}
    for name in test_stems:
        ds_path = Path(TEST_DIR) / name
        coords, edges = put.predict_video(
            model, ds_path, device, cfg=cfg,
            window_size=window_size, downsample=downsample,
        )
        graph = put.build_graph(coords, edges)
        geff_path = out_dir / f"{name}.geff"
        save_graph(graph, geff_path)
        candidate_geffs[det_thr][name] = geff_path
        print(f"det={det_thr} {name}: {graph.num_nodes()} nodes, {graph.num_edges()} edges", flush=True)

print("candidate graphs done")


## 4. ILP sweep (CPU) — one submission CSV per setting


In [ ]:
from pathlib import Path

import pandas as pd
import tracksdata as td

DIV_GRID = [0.2, 1.0]


def write_csv(graphs, csv_path):
    rows = []
    for name, graph in graphs.items():
        for r in graph.node_attrs().iter_rows(named=True):
            rows.append({
                "dataset": name, "row_type": "node", "node_id": int(r["node_id"]),
                "t": int(r["t"]), "z": int(round(r["z"])), "y": int(round(r["y"])),
                "x": int(round(r["x"])), "source_id": -1, "target_id": -1,
            })
        for r in graph.edge_attrs().iter_rows(named=True):
            rows.append({
                "dataset": name, "row_type": "edge", "node_id": -1,
                "t": -1, "z": -1, "y": -1, "x": -1,
                "source_id": int(r["source_id"]), "target_id": int(r["target_id"]),
            })
    df = pd.DataFrame(rows)
    df.insert(0, "id", range(len(df)))
    df.to_csv(csv_path, index=False)
    print(f"Wrote {csv_path} with {len(df)} rows", flush=True)


for det_thr, geffs in candidate_geffs.items():
    # Load candidate graphs once per det threshold.
    graphs = {}
    for name, gpath in geffs.items():
        graph = td.graph.IndexedRXGraph.from_geff(gpath)
        graphs[name] = graph[0] if isinstance(graph, tuple) else graph
    for div_w in DIV_GRID:
        solved = {}
        for name, graph in graphs.items():
            if graph.num_edges() == 0:
                solved[name] = graph
                continue
            solver = td.solvers.ILPSolver(
                edge_weight=-1.0 * td.EdgeAttr("edge_prob"),
                appearance_weight=0.1,
                disappearance_weight=0.1,
                division_weight=div_w,
            )
            with put.suppress_output():
                solved[name] = solver.solve(graph)
        csv_path = f"/kaggle/working/submission_det{det_thr:.3f}_div{div_w:.1f}.csv"
        write_csv(solved, csv_path)


## Submissions

Each `submission_det*_div*.csv` is a candidate. Submit them individually via the
competition API (`competition_submit_code`) — each reuses this run's output, so
no extra GPU is spent per submission.
